#### Удаление строк

Для удаления строк в `PostgreSQL` используется оператор `DELETE`:

```sql
DELETE FROM table_name
WHERE condition;
```

`Используя psycopg`

```python
import psycopg

try:
    conn = psycopg.connect(dbname = 'hexlet', user = 'sergey')
    print('соединение установлено')
except:
    print('None')
```
1. Удаление по условию одной строки (строка 9)

```python
cur = conn.cursor()
cur.execute("DELETE FROM orders WHERE id = %s", (9,))
conn.commit()
cur.close()
```

2. Удаление по условию двух и более строк используя `кортеж`

```python
cur = conn.cursor()
cur.execute(
    "DELETE FROM orders WHERE id IN (%s, %s)",
    (123, 124),
)
conn.commit()
cur.close()
```

3. Удаление используя `ANY` с массивом (удобно, когда количество неизвестно заранее)

```python
cur = conn.cursor()
cur.execute(
    "DELETE FROM orders WHERE id = ANY(%S)",
    ([9, 18],),
)
conn.commit()
cur.close()
```

4. Множественное удаление

```python
cur = conn.cursor()
cur.execute("DELETE FROM users WHERE status = %s AND deleted_at IS NOT NULL"
("blocked",))
conn.commit
cur.close
```
Этот пример показывает, что одна `DELETE-команда` может удалить сколько угодно строк, а не одну. Название «множественное» тут относится не к нескольким командам, а к тому, что условие WHERE охватывает целый набор строк.

`Разбор условия:`

```sql
"DELETE FROM users WHERE status = %s AND deleted_at IS NOT NULL"
("blocked",)
```
- `status = 'blocked'` — только заблокированные пользователи;
- `AND deleted_at IS NOT NULL` — и только те, у кого в поле `deleted_at` стоит дата (то есть их уже когда-то «мягко удалили», пометили к удалению).

Запрос удалит все строки, подходящие под оба условия сразу. Если под них попадает 0, 5 или 15 000 пользователей — команда одна и та же, удаление происходит одним запросом в базе.

Это типичный приём сопровождения БД: записи не удаляют сразу, а ставят флаг/дату `(deleted_at = now())`, а убирают физически позже пакетной очисткой — как раз такой `DELETE`. Один здесь плейсхолдер, потому что в SQL всего одно подставляемое значение `(blocked);` `IS NOT NULL` — это SQL-условие, ему значение не нужно.

Если коротко: «множественное» = одна команда удаляет целый набор строк, отобранных по WHERE. Отдельный DELETE по каждой строке не пишут.

---

##### `Удаление всех строк таблицы`

Команда `DELETE` без условия удалит все строки:

```sql
DELETE FROM имя_таблицы;
```

Это удаляет данные, но не сбрасывает счётчик автоинкремента `(SERIAL/IDENTITY)` — следующие вставленные записи продолжат нумерацию с прежнего значения.

`Быстрый способ для таблиц без внешних ключей-зависимостей (или с CASCADE):`

```sql
TRUNCATE TABLE имя_таблицы;
```

`TRUNCATE` работает быстрее и при необходимости сбрасывает счётчик:

```sql
TRUNCATE TABLE имя_таблицы RESTART IDENTITY;
```

`Если на таблицу ссылаются внешние ключи:`

```sql
TRUNCATE TABLE имя_таблицы CASCADE;
```

- `DELETE` — построчное удаление (можно в транзакции с WHERE, логируется полностью, триггеры срабатывают). 

- `TRUNCATE` — мгновенная очистка (нельзя с WHERE, транзакцию откатить можно, но нельзя применить к таблицам-родителям FK без `CASCADE`). 

- Для просто очистить все данные — `TRUNCATE` предпочтительнее.
---

##### `Подробнее про TRANCATE....CASCADE`

Насчёт `CASCADE` — небольшой нюанс: он удаляет и строки из других таблиц, у которых есть внешний ключ на предполагаемую таблицу `customers`. У вас это `orders`, так что эффект тот же, но `CASCADE` делает и больше (сработал бы на любые зависимости). Если хотите `truncate` именно `customers` и `orders` и ничего больше, безопаснее указать их явно без `CASCADE`:

```sql
cur.execute("TRUNCATE TABLE orders, customers RESTART IDENTITY")
```

Здесь порядок имеет значение: 
- сначала дочерняя `orders`, 
- потом родительская `customers`, иначе PostgreSQL заругается на внешний ключ. 
- a `RESTART IDENTITY` сбросит serial-счётчики в обеих таблицах, чтобы `ID` снова начинались с 1.